In [ ]:
import django
from io import StringIO
import os
import pandas
from pathlib import Path
import yaml


In [ ]:
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'mousedemo.settings')
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
django.setup()

from igvf_mice import models
from igvf_mice.io.converters import (
    str_or_empty,
    str_or_none,
)

In [ ]:
fixture_dir = Path("igvf_mice/fixtures")

In [ ]:
with open(fixture_dir/"source.yaml", "rt") as instream:
    sources = [x["pk"] for x in yaml.load(instream, Loader=yaml.SafeLoader)]

sources

In [ ]:
strain_urls = {
    "AJ": "https://www.jax.org/strain/000646",
    "B6J": "https://www.jax.org/strain/000664",
    "129S1J": "https://www.jax.org/strain/002448",
    "NODJ": "https://www.jax.org/strain/001976",
    "NZOJ": "https://www.jax.org/strain/002105",
    "CASTJ": "https://www.jax.org/strain/000928",
    "PWKJ": "https://www.jax.org/strain/003715",
    "WSBJ": "https://www.jax.org/strain/001145",
    'B6129S1F1J': "https://www.jax.org/strain/101043",
    'B6AF1J': "https://www.jax.org/strain/100002",
    'B6CASTF1J': None,
    'B6NODF1J': None,
    'B6NZOF1J': None,
    'B6PWKF1J': None,
    'B6WSBF1J': "https://www.jax.org/strain/019019",
    'TREM2R47HNSS_HO': 'https://www.jax.org/strain/034036',
    'CC001': "https://www.jax.org/strain/021238",
    'CC002': "https://www.jax.org/strain/021236",
    'CC003': "https://www.jax.org/strain/021237",
    'CC004': "https://www.jax.org/strain/020944",
    'CC005': "https://www.jax.org/strain/020945",
    'CC006': "https://www.jax.org/strain/022869",
    'CC007': "https://www.jax.org/strain/029625",
    'CC008': "https://www.jax.org/strain/026971",
    'CC009': "https://www.jax.org/strain/018856",
    'CC010': "https://www.jax.org/strain/021889",
    'CC011': "https://www.jax.org/strain/018854",
    'CC012': "https://www.jax.org/strain/028409",
    'CC013': "https://www.jax.org/strain/021892",
    'CC015': "https://www.jax.org/strain/018859",
    'CC017': "https://www.jax.org/strain/022870",
    'CC018': "https://www.jax.org/strain/021890",
    'CC024': "https://www.jax.org/strain/021891",
    'CC025': "https://www.jax.org/strain/018857",
    'CC028': "https://www.jax.org/strain/025126",
    'CC029': "https://www.jax.org/strain/026972",
    'CC030': "https://www.jax.org/strain/025426",
    'CC032': "https://www.jax.org/strain/020946",
    'CC036': "https://www.jax.org/strain/025127",
    'CC037': "https://www.jax.org/strain/025423",
    'CC038': None,
    'CC041': "https://www.jax.org/strain/021893",
    'CC055': None,
    'CC057': "https://www.jax.org/strain/024683",
    'CC060': "https://www.jax.org/strain/026427",
    'CC062': None,
    'CC065': None,
    'CC071': None,
    'CC074': "https://www.jax.org/strain/018855",
}

# used as name. Should this be 
strain_igvf_id = {
    "AJ": "A/J (AJ)",
    "B6J": "C57BL/6J (B6)",
    "129S1J": "129S1/SvImJ (129)",
    "NODJ": "NOD/ShiLtJ (NOD)",
    "NZOJ": "NZO/HlLtJ (NZO)",
    "CASTJ": "CAST/EiJ (CAST)",
    "PWKJ": "PWK/PhJ (PWK)",
    "WSBJ": "WSB/EiJ (WSB)",
    'B6129S1F1J': None, #"B6129SF1/J (B6 female x 129S male F1)"
    'B6AF1J':    None, #"B6AF1/J (B6 female x AJ male F1)"
    'B6CASTF1J': None, #"B6CASTF1/J (B6 female x CAST male F1)"
    'B6NODF1J':  None, #"B6NODF1/J (B6 female x NOD male F1)"
    'B6NZOF1J':  None, #"B6NZOF1/J (B6 female x NZO male F1)"
    'B6PWKF1J':  None, #"B6PWKF1/J (B6 female x PWK male F1)"
    'B6WSBF1J':  None, #"B6WSBF1/J (B6 female x WSB male F1)"
    "TREM2R47HNSS_HO": "B6(SJL)-Trem2<em1Aduci>/J", #"B6(SJL)-Tre2ᵉᵐ¹ᴬᵈᵘᶜⁱ/J",
    # commented out CC lines are ones we're not currently using.
    'CC001': 'CC001/Unc',
    'CC002': 'CC002/Unc',
    'CC003': 'CC003/Unc',
    'CC004': 'CC004/TauUnc',
    'CC005': 'CC005/TauUnc',
    'CC006': 'CC006/TauUnc',
    'CC007': 'CC007/Unc',
    'CC008': 'CC008/GeniUnc',
    'CC009': 'CC009/Unc',
    'CC010': 'CC010/GeniUnc',
    'CC011': 'CC011/Unc',
    'CC012': 'CC012/GeniUnc',
    'CC013': 'CC013/GeniUnc',
    'CC015': 'CC015/Unc',
    #'CC016': 'CC016/GeniUnc',
    'CC017': 'CC017/Unc',
    'CC018': 'CC018/Unc',
    #'CC019': 'CC019/TauUnc',
    #'CC020': 'CC020/GeniUnc',
    #'CC021': 'CC021/Unc',
    #'CC022': 'CC022/GeniUnc',
    #'CC023': 'CC023/GeniUnc',
    'CC024': 'CC024/GeniUnc',
    'CC025': 'CC025/GeniUnc',
    #'CC026': 'CC026/GeniUnc',
    #'CC027': 'CC027/GeniUnc',
    'CC028': 'CC028/GeniUnc',
    'CC029': 'CC029/Unc',
    'CC030': 'CC030/GeniUnc',
    #'CC031': 'CC031/GeniUnc',
    'CC032': 'CC032/GeniUnc',
    #'CC033': 'CC033/GeniUnc',
    #'CC034': 'CC034/Unc',
    #'CC035': 'CC035/Unc',
    'CC036': 'CC036/Unc',
    'CC037': 'CC037/TauUnc',
    'CC038': 'CC038/GeniUnc',
    #'CC039': 'CC039/Unc',
    #'CC040': 'CC040/TauUnc',
    'CC041': 'CC041/TauUnc',
    #'CC042': 'CC042/GeniUnc',
    'CC043': 'CC043/GeniUnc',
    #'CC044': 'CC044/Unc',
    #'CC045': 'CC045/GeniUnc',
    #'CC046': 'CC046/Unc',
    #'CC047': 'CC047/Unc',
    #'CC048': 'CC048/Unc',
    #'CC049': 'CC049/TauUnc',
    #'CC050': 'CC050/Unc',
    #'CC051': 'CC051/TauUnc',
    #'CC052': 'CC052/GeniUnc',
    #'CC053': 'CC053/Unc',
    #'CC054': 'CC054/GeniUnc',
    'CC055': 'CC055/TauUnc',
    #'CC056': 'CC056/GeniUnc',
    'CC057': 'CC057/Unc',
    #'CC058': 'CC058/Unc',
    #'CC059': 'CC059/TauUnc',
    'CC060': 'CC060/Unc',
    #'CC061': 'CC061/GeniUnc',
    'CC062': 'CC062/Unc',
    #'CC063': 'CC063/Unc',
    'CC065': 'CC065/Unc',
    #'CC068': 'CC068/TauUnc',
    #'CC070': 'CC070/TauUnc',
    'CC071': 'CC071/TauUnc',
    #'CC072': 'CC072/TauUnc',
    #'CC073': 'CC073/Unc',
    'CC074': 'CC074/Unc',
    #'CC075': 'CC075/Unc',
    #'CC076': 'CC076/Unc',
    #'CC078': 'CC078/TauUnc',
    #'CC079': 'CC079/TauUnc',
    #'CC080': 'CC080/TauUnc',
    #'CC081': 'CC081/Unc',
}

def int_csv_to_hex(x):
    r, g, b = x.split(",")
    
    return "#{:02x}{:02x}{:02x}".format(int(r), int(g), int(b))

def remove_citation(x):
    return x.split(" ")[0]

#strains = pandas.read_excel(
#    lizs_sheet_name, 
#    sheet_name="Line information", 
#    usecols=[
#        "Designation",
#        "Strain",
#        "Note",
#        "Jax Catalog No",
#        "Sample CODE",
#        "Strain notes",
#    ],
#    converters={
#        "Strain": remove_citation,
#        "Jax Catalog No": str,
#        "Strain notes": str_or_empty,
#    }
#).dropna(how="all")
# Remove the extra lines with the bridge samples for now. I don't want to track thta level of detail
#bridge = strains[strains["Designation"] == "Bridge sample"].first_valid_index() - 1
#strains = strains.iloc[0:bridge]

# Add in Strain URL
#strains["see_also"] = strains["Sample CODE"].apply(lambda x: strain_urls.get(x, x))
#strains

#print(strains.to_csv(index=False))

strains_csv = StringIO("""Designation,Strain,Note,Jax Catalog No,Sample CODE,Strain notes,see_also,Source
A,AJ,CC founder,000646,AJ,Yellower adrenal gland.,https://www.jax.org/strain/000646,jackson-labs
B,C57BL/6J,CC founder,000664,B6J,,https://www.jax.org/strain/000664,jackson-labs
C,129S1/SvImJ,CC founder,002448,129S1J,Male skin is tougher than female skin. Gallbladder is more filled in this strain. Adrenals are pale and smaller than A6J.,https://www.jax.org/strain/002448,jackson-labs
D,NOD/ShiLtJ,CC founder,001976,NODJ,,https://www.jax.org/strain/001976,jackson-labs
E,NZO/HlLtJ,CC founder,002105,NZOJ,,https://www.jax.org/strain/002105,jackson-labs
F,CAST/EiJ,CC founder,000928,CASTJ,,https://www.jax.org/strain/000928,jackson-labs
G,PWK/PhJ,CC founder,003715,PWKJ,,https://www.jax.org/strain/003715,jackson-labs
H,WSB/EiJ,CC founder,001145,WSBJ,,https://www.jax.org/strain/001145,jackson-labs
,B6129S1F1/J,CC F1,101043,B6129S1F1J,,https://www.jax.org/strain/101043,jackson-labs
,B6AF1/J,CC F1,101043,B6AF1J,,https://www.jax.org/strain/100002,jackson-labs
,B6CASTF1/J,CC F1,Custom C56BL/6J crossed CAST/EiJ,B6CASTF1J,,,jackson-labs
,B6NODF1/J,CC F1,Custom C56BL/6J crossed NOD/ShiLtJ,B6NODF1J,,,jackson-labs
,B6NZOF1/J,CC F1,Custom C56BL/6J crossed NZO/HlLtJ,B6NZOF1J,,,jackson-labs
,B6PWKF1/J,CC F1,Custom C56BL/6J crossed PWK/PhJ,B6PWKF1J,,,jackson-labs
,B6WSBF1/J,CC F1,019019,B6WSBF1J,,https://www.jax.org/strain/019019,jackson-labs
,TREM2R47HNSS_HO,CC Mutant,033781,TREM2R47HNSS_HO,,https://www.jax.org/strain/033781,jackson-labs
,CC001,CC Cross,021238,CC001,,https://www.jax.org/strain/021238,unc-csbio
,CC002,CC Cross,021236,CC002,,https://www.jax.org/strain/021236,unc-csbio
,CC003,CC Cross,021237,CC003,,https://www.jax.org/strain/021237,unc-csbio
,CC004,CC Cross,020944,CC004,,https://www.jax.org/strain/020944,unc-csbio
,CC005,CC Cross,020945,CC005,,https://www.jax.org/strain/020945,unc-csbio
,CC006,CC Cross,022869,CC006,,https://www.jax.org/strain/022869,unc-csbio
,CC007,CC Cross,029625,CC007,,https://www.jax.org/strain/029625,unc-csbio
,CC008,CC Cross,026971,CC008,,https://www.jax.org/strain/026971,unc-csbio
,CC009,CC Cross,018856,CC009,,https://www.jax.org/strain/018856,unc-csbio
,CC010,CC Cross,021889,CC010,,https://www.jax.org/strain/021889,unc-csbio
,CC011,CC Cross,018854,CC011,,https://www.jax.org/strain/018854,unc-csbio
,CC012,CC Cross,028409,CC012,,https://www.jax.org/strain/028409,unc-csbio
,CC013,CC Cross,021892,CC013,,https://www.jax.org/strain/021892,unc-csbio
,CC015,CC Cross,018859,CC015,,https://www.jax.org/strain/018859,unc-csbio
,CC017,CC Cross,022870,CC017,,https://www.jax.org/strain/022870,unc-csbio
,CC018,CC Cross,021890,CC018,,https://www.jax.org/strain/021890,unc-csbio
,CC024,CC Cross,021891,CC024,,https://www.jax.org/strain/021891,unc-csbio
,CC025,CC Cross,018857,CC025,,https://www.jax.org/strain/018857,unc-csbio
,CC028,CC Cross,025126,CC028,,https://www.jax.org/strain/025126,unc-csbio
,CC029,CC Cross,026972,CC029,,https://www.jax.org/strain/026972,unc-csbio
,CC030,CC Cross,025426,CC030,,https://www.jax.org/strain/025426,unc-csbio
,CC032,CC Cross,020946,CC032,,https://www.jax.org/strain/020946,unc-csbio
,CC036,CC Cross,025127,CC036,,https://www.jax.org/strain/025127,unc-csbio
,CC037,CC Cross,025423,CC037,,https://www.jax.org/strain/025423,unc-csbio
,CC038,CC Cross,,CC038,,,unc-csbio
,CC041,CC Cross,021893,CC041,,https://www.jax.org/strain/021893,unc-csbio
,CC043,CC Cross,023828,CC043,,https://www.jax.org/strain/023828,unc-csbio
,CC055,CC Cross,,CC055,,,unc-csbio
,CC057,CC Cross,024683,CC057,,https://www.jax.org/strain/024683,unc-csbio
,CC060,CC Cross,026427,CC060,,https://www.jax.org/strain/026427,unc-csbio
,CC062,CC Cross,,CC062,,,unc-csbio
,CC065,CC Cross,,CC065,,,unc-csbio
,CC071,CC Cross,,CC071,,,unc-csbio
,CC074,CC Cross,018855,CC074,,https://www.jax.org/strain/018855,unc-csbio""")

strains = pandas.read_csv(
    strains_csv,
    converters={"Jax Catalog No": str_or_none}
)

for i, row in strains.iterrows():
    assert pandas.notnull(row["Strain"]), f"{row['Strain']} was null {i} {row}"
    assert pandas.notnull(row["Sample CODE"]), f"{row['Sample CODE']} was null {i} {row}"
    assert pandas.notnull(row["Source"]), f"{row['Source']} was null {i} {row}"
    assert row["Source"] in sources

strains


In [ ]:
str(models.StrainType.FOUNDER)

In [ ]:
strain_type_lookup = {
    "CC founder": str(models.StrainType.FOUNDER),
    "CC F1": str(models.StrainType.F1),
    "CC Cross": str(models.StrainType.CROSS),
    "CC Mutant": str(models.StrainType.MUTANT),
}


# this version is from my embedded csv file, so less adjusting needed
fixtures = []
for i, row in strains.iterrows():
    fixtures.append({
        "model": "igvf_mice.mousestrain",
        "pk": row["Sample CODE"],
        "fields": {
            "display_name": row["Strain"],
            "igvf_id": strain_igvf_id[row["Sample CODE"]] if pandas.notnull(row["Sample CODE"]) else None,
            "strain_type": strain_type_lookup[row["Note"]],
            "jax_catalog_number": str_or_none(row["Jax Catalog No"]),
            "notes": str_or_empty(row["Strain notes"]),
            "see_also": str_or_none(row["see_also"]),
            "source": row["Source"],
        }
    })
    
print(yaml.dump(fixtures))

In [ ]:
with open(fixture_dir / "mousestrain.yaml", "wt") as outstream:
    outstream.write(yaml.dump(fixtures))